# Rag Intro

- Retrieval Augmented Generation (RAG) is an AI framework that improves the output of large language models (LLMs) by providing them with new, relevant, and authoritative information from external knowledge sources.

- Instead of relying solely on their pre-trained data, which can be outdated or inaccurate, a RAG system first retrieves relevant information and then uses it to "augment" the user's query, giving the LLM a more complete and factual context from which to generate a response.

- This process helps to reduce the problem of hallucinations, where LLMs create plausible but incorrect information.

## Phases to Build and Use a RAG System

Building a RAG system can be broken down into two main phases:

### 1. Indexing Phase (Data Preparation)

- This phase is about preparing your external knowledge base so that it can be easily searched and retrieved from later on.

    - **Data Loading and Chunking:** Gather your documents, articles, databases, or any other data you want the system to reference. Since LLMs have a limited context window, you must break these large documents into smaller, more manageable pieces, or chunks.

    - **Creating Embeddings:** An embedding model converts these text chunks into numerical representations called vectors. These vectors capture the semantic meaning of the text, allowing the system to understand the context and relationships between different pieces of information.

    - **Storing in a Vector Database:** The vectors are then stored in a specialized database, often a vector database, which is optimized for fast and efficient similarity searches. This database acts as your knowledge library for the retrieval phase.

### 2. Retrieval and Generation Phase

- This phase is the runtime process that happens when a user submits a query.

    - **Retrieval:** When a user asks a question, their query is also converted into a vector. The system then uses this query vector to perform a similarity search in the vector database, identifying and retrieving the most relevant document chunks that semantically match the question.

    - **Augmentation and Generation:** The retrieved chunks are combined with the original user query to create an augmented prompt. This enriched prompt is then fed into the LLM. The LLM uses this new, factual context, along with its own internal knowledge, to generate a coherent, accurate, and relevant answer to the user's question.

- It's advisable to use a very good LLM for the generation phase, as the quality of the response will heavily depend on the model's capabilities.

In [1]:
import json
import os
import warnings
from pathlib import Path
from typing import Any, Generator, Iterable, Type, TypeVar

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = "google/gemini-2.0-flash-001"
model_str_local: str = (
    "mistral:7b-instruct-v0.3-q4_0"  # llama3.1:8b, mistral:7b-instruct-v0.3-q4_0, llama3.2:3b
)

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

<br>

## Vector Store And Embeddings

- TBC


### Create A QDRANT Client

In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(url=settings.QDRANT_URL)
emb_model = OllamaEmbeddings(
    model="mxbai-embed-large:latest",
)
emb = emb_model.embed_documents("Hello world")
emb_size: int = len(emb[0])

# Create a Qdrant collection (if it doesn't exist)
# client.create_collection(
#     collection_name="demo_collection",
#     vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
# )

# Recreate the collection to ensure it's fresh
# client.recreate_collection(
#     collection_name="demo_collection",
#     vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
# )

# # Create a vector store using Qdrant
# vector_store = QdrantVectorStore(
#     client=client,
#     collection_name="demo_collection",
#     embedding=emb_model,
# )

### Add Documents To The Vectorstore

In [6]:
# from uuid import uuid4

# from langchain_core.documents import Document

# docs: list[str] = [
#     "I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
#     "The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees Fahrenheit.",
#     "Building an exciting new project with LangChain - come check it out!",
#     "It is good to be good",
# ]
# all_metadata: list[dict[str, Any]] = [
#     {"source": "tweet"},
#     {"source": "news"},
#     {"source": "tweet"},
#     {"source": "tweet"},
# ]
# documents: list[Document] = [Document(page_content=doc, metadata=meta) for doc, meta in zip(docs, all_metadata)]
# uuids = [str(uuid4()) for _ in range(len(documents))]

# vector_store.add_documents(documents=documents, ids=uuids)

In [7]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI

emb_model = OllamaEmbeddings(
    model="mxbai-embed-large:latest",
)
#### INDEXING ####

# Load Documents
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))),
# )
loader = PyPDFLoader(file_path="../../data/chelsea_transfer_news.pdf")
docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1_000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
len(splits)

25

In [9]:
console.print(docs[0])

Document(
    metadata={
        'producer': 'Skia/PDF m138',
        'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
        'creationdate': '2025-07-25T18:28:43+00:00',
        'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports',
        'moddate': '2025-07-25T18:28:43+00:00',
        'source': '../../data/chelsea_transfer_news.pdf',
        'total_pages': 12,
        'page': 0,
        'page_label': '1'
    },
    page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates and 
latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: Chelsea 
2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the Sky 
Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naround the deal.\xa0\nKeep scrolling!\xa0\nUPDATE\nF o o t b al l
\n News Watch Scores & FixturesTables Transfers More\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: 
Live updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 1/17'
)

In [10]:
from uuid import uuid4

collection_name: str = "demo_collection_2"

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
)

# Create a vector store using Qdrant
vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=emb_model,
    collection_name=collection_name,
    ids=[str(uuid4()) for _ in range(len(splits))],
)

In [11]:
prompt = hub.pull("rlm/rag-prompt")
console.print(prompt)

ChatPromptTemplate(
    input_variables=['context', 'question'],
    input_types={},
    partial_variables={},
    metadata={
        'lc_hub_owner': 'rlm',
        'lc_hub_repo': 'rag-prompt',
        'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'
    },
    messages=[
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['context', 'question'],
                input_types={},
                partial_variables={},
                template="You are an assistant for question-answering tasks. Use the following pieces of retrieved 
context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences 
maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"
            ),
            additional_kwargs={}
        )
    ]
)

In [12]:
llm = local_llm  # or remote_llm

retriever = vector_store.as_retriever()

### Indexing

[![image.png](https://i.postimg.cc/NFdTN4vj/image.png)](https://postimg.cc/vx6cT5mJ)

In [13]:
from langchain_core.runnables import RunnableMap


def format_docs(docs: list[Any]) -> str:
    """Append the page content of each document into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)


debugging_rag_chain = (
    RunnableMap(
        {
            "context": retriever
            | format_docs,  # Ensure retriever and format_docs are chained correctly
            "question": RunnablePassthrough(),
        }
    )
    | (
        lambda x: console.print(f"DEBUG: Context: {x['context']}") or x
    )  # Debugging print
    | prompt
    | llm
    | {"response": lambda x: x.content}
)

In [14]:
query: str = "Who is Xavi and what club does he play for?"
response = debugging_rag_chain.invoke(query)
console.print(response)

DEBUG: Context: "Xavi chose the move to Leipzig wisely, together with his camp. The goal was
to take on a significant role at an ambitious, up-and-coming club. 
"Especially in his first year, he performed brilliantly — in great form, he became
a standout both at Leipzig and in the Bundesliga. 
"The permanent transfer to Leipzig had been discussed well in advance; it was
no surprise. His contract, running until 2027, clearly indicates: Leipzig is not
meant to be the final step or final club. 
"Xavi sees Leipzig and the Bundesliga as a major opportunity for
development. Now, together with his camp, he wants to take the next step up
the ladder. The Bundesliga has been good for him."
24 Jul
16:06
How did Simons perform last season?
Sky Germany’s RB Leipzig reporter Philipp Hinze: 
"Xavi’s last season was definitely not as poor as it initially seemed. His stats
were totally fine; he contributed several goal involvements. Unfortunately, a

"Xavi’s last season was definitely not as poor as it initially seemed. His stats
were totally fine; he contributed several goal involvements. Unfortunately, a
serious injury held him back — he tore his syndesmosis ligament and was
sidelined for several months. 
"Xavi sees himself as a leader at Leipzig. He often puts too much pressure on
himself. In the end, the entire club disappointed. It was the worst Bundesliga
season in the club’s history. 
"As a result, Xavi’s performance was also described as insufficient. The whole
team underperformed badly. But in truth, Xavi’s attacking numbers were
actually solid.
"The club’s poor season casts a heavy shadow over all the players. After a very
strong season the year before, Xavi missed the chance to take the next step
in his development."
Advertisement
25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports

the camp. 
"A normal situation — Xavi is still a Leipzig player and is training regularly with
the team without any restrictions. He is behaving positively and is in good
spirits. There are no signs of a bad mood. Xavi is showing good form in
training."
24 Jul
15:55
Isak not a target for Chelsea
Latest from Sky Sports News' Kaveh Solhekol:
25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports
https://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-updates-a
nd-latest-on-deals-signings-loans-and-… 6/17

the two know each other very well. 
"Recently, Bayern contacted the player’s side by phone to gather information.
Bayern should definitely not be completely ruled out in the race for Xavi. They
are still keeping a low profile in the background. Chelsea are currently leading
the race. 
"We’ll have to wait and see what happens — and whether Bayern become
more active if Chelsea don’t close the deal. The agency EPIC has come on
board to help make a transfer away from Leipzig possible for Xavi this
summer. An exciting story."
24 Jul
16:07
How has Simons adapted in the Bundesliga?
Sky Germany’s RB Leipzig reporter Philipp Hinze: 
25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports
https://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-updates-a
nd-latest-on-deals-signings-loans-and-… 4/17

{
    'response': ' Xavi is a player for RB Leipzig in the Bundesliga. Last season, his performance was not as poor 
as it initially seemed due to a serious injury that held him back. Despite a disappointing team performance 
overall, his attacking numbers were solid. Currently, there are rumors about Chelsea and Bayern Munich being 
interested in transferring Xavi.'
}

### Count Tokens

- A token is ~4 characters.


In [15]:
import tiktoken


def num_tokens_from_string(input_text: str, encoding_name: str) -> int:
    """Returns the number of tokens in a string."""
    encoding = tiktoken.get_encoding(encoding_name)
    tokens = encoding.encode(input_text)
    return len(tokens)

In [16]:
document: str = """
Xavi felt responsible for Leipzig’s worst Bundesliga season, yet his attacking stats remained solid. After a bright 
first year, the club’s slump stalled his progress. Despite a 2027 contract, he and his camp now view Leipzig as a 
stepping-stone and are seeking a bigger move while he trains normally
"""
num_tokens = num_tokens_from_string(query, "cl100k_base")
print(f"Query: {query}")
print(f"Number of tokens: {num_tokens}")

Query: Who is Xavi and what club does he play for?
Number of tokens: 12


In [17]:
num_tokens_from_string("Xavi", "cl100k_base")

2

In [18]:
query_result = emb_model.embed_query(query)
document_result = emb_model.embed_query(document)
len(query_result)

1024

#### Calculate Similarity

- Using `Cosine Similarity`

In [19]:
def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product / (norm_a * norm_b) if norm_a and norm_b else 0.0

In [20]:
similarity = cosine_similarity(query_result, document_result)
console.print(f"Cosine similarity: {similarity}")

Cosine similarity: 0.7208322701050109

<br>

### Chunking And Indexing

- This text splitter is the recommended one for generic text. 

- It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. 

- The default list is ["\n\n", "\n", " ", ""]. 

- This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [21]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)

# Make splits
splits = text_splitter.split_documents(docs)
len(splits)

22

In [22]:
collection_name: str = "demo_collection_2"

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
)

# Create a vector store using Qdrant
vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=emb_model,
    collection_name=collection_name,
    ids=[str(uuid4()) for _ in range(len(splits))],
)

retriever = vector_store.as_retriever(search_kwargs={"k": 1})

In [23]:
docs = retriever.get_relevant_documents(query)
len(docs)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_34510/1850314445.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


1

In [24]:
console.print(docs)

[
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 4,
            'page_label': '5',
            '_id': 'f232d436-dcae-496d-b054-6b1720c6c805',
            '_collection_name': 'demo_collection_2'
        },
        page_content='"Xavi chose the move to Leipzig wisely, together with his camp. The goal was\nto take on a 
significant role at an ambitious, up-and-coming club.\xa0\n"Especially in his first year, he performed brilliantly 
— in great form, he became\na standout both at Leipzig and in the Bundesliga.\xa0\n"The permanent transfer to 
Leipzig had been discussed well in advance; it was\nno surprise. His contract, running until 2027, clearly 
indicates: Leipzig is not\nmeant to be the final step or final club.\xa0\n"Xavi sees Leipzig and the Bundesliga as 
a major opportunity for\ndevelopment. Now, together with his camp, he wants to take the next step up\nthe ladder. 
The Bundesliga has been good for him."\n24 Jul\n16:06\nHow did Simons perform last season?\nSky Germany’s RB 
Leipzig reporter Philipp Hinze:\xa0\n"Xavi’s last season was definitely not as poor as it initially seemed. His 
stats\nwere totally fine; he contributed several goal involvements. Unfortunately, a\nserious injury held him back 
— he tore his syndesmosis ligament and was\nsidelined for several months.\xa0\n"Xavi sees himself as a leader at 
Leipzig. He often puts too much pressure on'
    )
]

### Generation


[![image.png](https://i.postimg.cc/wxJ2kKr1/image.png)](https://postimg.cc/nMFqHWpx)

In [25]:
from langchain.prompts import ChatPromptTemplate

template: str = """
<user>
Answer the question based only on the following context:
<context>{context}</context>
<question>{question}</question>

<response_guide>
- Always respond in a concise and informative manner with a maximum of 3 sentences.
</response_guide>

</user>
"""

prompt = ChatPromptTemplate.from_template(template)
console.print(prompt)

ChatPromptTemplate(
    input_variables=['context', 'question'],
    input_types={},
    partial_variables={},
    messages=[
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['context', 'question'],
                input_types={},
                partial_variables={},
                template='\n<user>\nAnswer the question based only on the following 
context:\n<context>{context}</context>\n<question>{question}</question>\n\n<response_guide>\n- Always respond in a 
concise and informative manner with a maximum of 3 sentences.\n</response_guide>\n\n</user>\n'
            ),
            additional_kwargs={}
        )
    ]
)

In [26]:
console.print(format_docs(docs[:2]))

"Xavi chose the move to Leipzig wisely, together with his camp. The goal was
to take on a significant role at an ambitious, up-and-coming club. 
"Especially in his first year, he performed brilliantly — in great form, he became
a standout both at Leipzig and in the Bundesliga. 
"The permanent transfer to Leipzig had been discussed well in advance; it was
no surprise. His contract, running until 2027, clearly indicates: Leipzig is not
meant to be the final step or final club. 
"Xavi sees Leipzig and the Bundesliga as a major opportunity for
development. Now, together with his camp, he wants to take the next step up
the ladder. The Bundesliga has been good for him."
24 Jul
16:06
How did Simons perform last season?
Sky Germany’s RB Leipzig reporter Philipp Hinze: 
"Xavi’s last season was definitely not as poor as it initially seemed. His stats
were totally fine; he contributed several goal involvements. Unfortunately, a
serious injury held him back — he tore his syndesmosis ligament and was
sidelined for several months. 
"Xavi sees himself as a leader at Leipzig. He often puts too much pressure on

In [27]:
simple_chain = prompt | llm
response = simple_chain.invoke({"context": format_docs(docs), "question": query})
console.print(response)

AIMessage(
    content=' Xavi is a football player who currently plays for RB Leipzig.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 16,
            'prompt_tokens': 380,
            'total_tokens': 396,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-539',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--d8325fb9-0de8-4b56-9607-6f6ec7e29a97-0',
    usage_metadata={
        'input_tokens': 380,
        'output_tokens': 16,
        'total_tokens': 396,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [28]:
temporary_chain = (
    RunnableMap(
        {
            "context": retriever | format_docs,  # Chain retriever and format_docs
            "question": RunnablePassthrough(),
        }
    )
    # | (lambda x: console.print(f"DEBUG \n===== \nContext: {x['context']}\n====") or x) # For debugging
    | prompt  # Use the prompt template
    | llm  # Pass the result to the language model
)

In [29]:
query: str = "Who is Xavi and what club does he play for?"
response = temporary_chain.invoke(query)
console.print(response)

AIMessage(
    content=' Xavi is a football player who currently plays for RB Leipzig.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 16,
            'prompt_tokens': 380,
            'total_tokens': 396,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-47',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--c32f9a9d-3e28-4b6f-b868-0661a2b6211d-0',
    usage_metadata={
        'input_tokens': 380,
        'output_tokens': 16,
        'total_tokens': 396,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [30]:
query: str = "Any news about Sterling?"
response = temporary_chain.invoke(query)
console.print(response)

AIMessage(
    content=" Yes, there is news about Sterling. According to the BBC, Fulham have expressed an interest in signing
Raheem Sterling from Chelsea this summer. It's suggested that Sterling has no future at Stamford Bridge and is 
among a group of players surplus to requirements.",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 61,
            'prompt_tokens': 397,
            'total_tokens': 458,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-699',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--c3301ede-10b0-4c09-9592-2954d956a2a4-0',
    usage_metadata={
        'input_tokens': 397,
        'output_tokens': 61,
        'total_tokens': 458,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [31]:
query: str = "What players are Chelsea FC interested in signing?"
response = temporary_chain.invoke(query)
console.print(response)

AIMessage(
    content=' Chelsea FC is reportedly interested in signing Xavi Simons from RB Leipzig and Carney Chukwuemeka, 
who currently plays for Chelsea but is being pursued by RB Leipzig. However, any further signings are dependent on 
players also leaving the club.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 60,
            'prompt_tokens': 374,
            'total_tokens': 434,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-765',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--6a2b667d-191a-4c31-8ec8-9c2d6fadac31-0',
    usage_metadata={
        'input_tokens': 374,
        'output_tokens': 60,
        'total_tokens': 434,
        'input_token_details': {},
        'output_token_details': {}
    }
)

### Using A Custom Prompt From The Hub

In [32]:
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")
console.print(prompt)

ChatPromptTemplate(
    input_variables=['context', 'question'],
    input_types={},
    partial_variables={},
    metadata={
        'lc_hub_owner': 'rlm',
        'lc_hub_repo': 'rag-prompt',
        'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'
    },
    messages=[
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['context', 'question'],
                input_types={},
                partial_variables={},
                template="You are an assistant for question-answering tasks. Use the following pieces of retrieved 
context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences 
maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"
            ),
            additional_kwargs={}
        )
    ]
)

In [33]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    # | {"response": lambda x: x.content}
)

In [34]:
query: str = "Who is Xavi and what club does he play for?"
response = rag_chain.invoke(query)
console.print(response)

AIMessage(
    content=' Xavi plays for RB Leipzig in the Bundesliga. Last season, despite a serious injury that sidelined him
for several months, his stats were fine and he contributed several goal involvements.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 42,
            'prompt_tokens': 376,
            'total_tokens': 418,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-718',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--090a1521-a4f8-4c1e-a475-fae2b848269c-0',
    usage_metadata={
        'input_tokens': 376,
        'output_tokens': 42,
        'total_tokens': 418,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [35]:
query: str = "Any news about Sterling?"
response = rag_chain.invoke(query)
console.print(response)

AIMessage(
    content=' Sterling is among a group of players at Chelsea who have no future under the current manager. Fulham 
has expressed interest in signing him this summer. Sterling still has two years left on his contract at Stamford 
Bridge.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 48,
            'prompt_tokens': 392,
            'total_tokens': 440,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-239',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--754b6fee-5cb6-4a33-861e-11574beb0267-0',
    usage_metadata={
        'input_tokens': 392,
        'output_tokens': 48,
        'total_tokens': 440,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [ ]:
query: str = "What players are Chelsea FC interested in signing?"
response = rag_chain.invoke(query)
console.print(response)